# pandas: apply

The ability to apply an arbitrary function across a `Series` or `DataFrame` axis in order to operate on a column, row, or element is a powerful feature of pandas.

In this lesson I'll demonstrate how to leverage the `Series.apply()`, `DataFrame.apply()` and `DataFrameGroupBy.apply()` methods as we continue to explore the Mega Millions lottery data covering the period 2017-2024.

💡 I've adjusted the `IPython` interactive shell's `ast_node_interactivity` setting to ensure that all cell output is displayed. This overrides the default last expression only ("last_expr") display behavior.

In [23]:
import altair as alt
import IPython as ipy
import pandas as pd


# Ensure that all interactive cell output is displayed (default="last_expr")
ipy.get_ipython().ast_node_interactivity = "all"

## Retrieve and prep the data

Let's read in the winning numbers data, convert the "draw_date" column dtype from `object` to `datetime64[ns]`, review a summary of the new `DataFrame` named `data`, and inspect its first three rows.

In [24]:
data = pd.read_csv("./data/mega_millions-transform-2017_24.csv")
data["draw_date"] = pd.to_datetime(data["draw_date"], format="%Y-%m-%d")

data.info()
data.head(3)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 684 entries, 0 to 683
Data columns (total 19 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   draw_date         684 non-null    datetime64[ns]
 1   draw_num          684 non-null    int64         
 2   winning_numbers   684 non-null    object        
 3   mega_ball         684 non-null    int64         
 4   mega_ball_mod2    684 non-null    int64         
 5   multiplier        684 non-null    int64         
 6   pick5_1           684 non-null    int64         
 7   pick5_1_zscore    684 non-null    float64       
 8   pick5_2           684 non-null    int64         
 9   pick5_2_zscore    684 non-null    float64       
 10  pick5_3           684 non-null    int64         
 11  pick5_3_zscore    684 non-null    float64       
 12  pick5_4           684 non-null    int64         
 13  pick5_4_zscore    684 non-null    float64       
 14  pick5_5           684 non-

,draw_date,draw_num,winning_numbers,mega_ball,mega_ball_mod2,multiplier,pick5_1,pick5_1_zscore,pick5_2,pick5_2_zscore,pick5_3,pick5_3_zscore,pick5_4,pick5_4_zscore,pick5_5,pick5_5_zscore,pick5_sum,pick5_sum_zscore,pick5_mod2_sum
0,2024-05-17,40,08 17 40 60 70,3,1,2,8,-0.359521,17,-0.492509,40,0.431437,60,1.082709,70,1.123843,195,0.495525,1
1,2024-05-14,39,13 19 43 62 64,6,0,3,13,0.214504,19,-0.321895,43,0.664474,62,1.241801,64,0.537490,201,0.632157,3
2,2024-05-10,38,13 22 26 32 65,18,0,4,13,0.214504,22,-0.065975,26,-0.656070,32,-1.144578,65,0.635216,158,-0.347041,2


## Apply

The [`DataFrame.apply()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.apply.html) and [`Series.apply()`](https://pandas.pydata.org/docs/reference/api/pandas.Series.apply.html) methods let you apply an arbitrary function along an axis. Only the `func` argument is required; all other arguments are optional. Both the `args` and `kwargs` parameters increase the flexibility of the method call by permitting the caller to pass additional positional or keyword arguments to the function bound to `func`.

`DataFrame.apply(func, axis=0, raw=False, result_type=None, args=(), by_row='compat', engine='python', engine_kwargs=None, **kwargs)`

`Series.apply(func, convert_dtype=_NoDefault.no_default, args=(), *, by_row='compat', **kwargs)`

### `Series.apply()`

In the example below, the `Series.apply()` method is called to apply the function named `get_month` to each "draw_date" element. The apply operation extracts either the number or name of the month.

In [25]:
def get_month(timestamp, name=False):
    """Access the month number or name from the passed in < timestamp >.

    Parameters
        timestamp (pandas.Timestamp): Timestamp object.
        name (bool): name flag.

    Returns
        int|str: Month number or name.
    """

    return timestamp.month_name() if name else timestamp.month

In [26]:
months = data["draw_date"].apply(get_month)
months

0       5
1       5
2       5
3       5
4       5
       ..
679    11
680    11
681    11
682    11
683    10
Name: draw_date, Length: 684, dtype: int64

In [27]:
months = data["draw_date"].apply(get_month, name=True)  # **kwargs
months

0           May
1           May
2           May
3           May
4           May
         ...   
679    November
680    November
681    November
682    November
683     October
Name: draw_date, Length: 684, dtype: object

Before opting for the `apply()` method I recommend that you consider carefully whether or not existing pandas functionality can satisfy your needs. In other words, avoid duplicating existing functionality.

In [28]:
months = data["draw_date"].dt.month
months

0       5
1       5
2       5
3       5
4       5
       ..
679    11
680    11
681    11
682    11
683    10
Name: draw_date, Length: 684, dtype: int32

In [29]:
months = data["draw_date"].dt.month_name()
months

0           May
1           May
2           May
3           May
4           May
         ...   
679    November
680    November
681    November
682    November
683     October
Name: draw_date, Length: 684, dtype: object

That said, this line of code illustrates the best of both worlds.

In [30]:
months = data["draw_date"].dt.month_name().apply(lambda x: x[:3])
months.unique()

array(['May', 'Apr', 'Mar', 'Feb', 'Jan', 'Dec', 'Nov', 'Oct', 'Sep',
       'Aug', 'Jul', 'Jun'], dtype=object)

But then again ...

In [31]:
months = data["draw_date"].dt.month_name().str[:3]
months.unique()

array(['May', 'Apr', 'Mar', 'Feb', 'Jan', 'Dec', 'Nov', 'Oct', 'Sep',
       'Aug', 'Jul', 'Jun'], dtype=object)

### `DataFrame.apply()`

 The previous example illustrated how to leverage the `Series.apply()` method when operating on a single `DataFrame` column (the pandas team describes the `DataFrame` "as a `dict`-like container for `Series` objects").

You can also call the `DataFrame.apply()` method to apply a function along an axis. The following example applies a `lambda` function to each `data` row in order to determine each draw's winning numbers range (i.e., the difference between the "pick5_5" (high) and "pick5_1" (low) values). The return value is a `Series` assigned to the variable named "pick5_range".

In [32]:
pick5_range = data.apply(lambda x: x["pick5_5"] - x["pick5_1"], axis=1)
pick5_range.info()

<class 'pandas.core.series.Series'>
RangeIndex: 684 entries, 0 to 683
Series name: None
Non-Null Count  Dtype
--------------  -----
684 non-null    int64
dtypes: int64(1)
memory usage: 5.5 KB


Currently, the "pick5_range" `Series` is unnamed. I'd like to join the `Series` to `data` but the lack of an index label guarantees that the join operation will trigger a `ValueError: Other Series must have a name`.

I can provide a label by calling the [`Series.rename()`](https://pandas.pydata.org/docs/reference/api/pandas.Series.rename.html) method.

In [33]:
pick5_range = pick5_range.rename("pick5_range")  # Comment out to trigger a ValueError
pick5_range.name

'pick5_range'

With the name added, I can join the `Series` to the `DataFrame` without triggering a runtime exception.

In [34]:
data = data.join(pick5_range)  # index-on-index join
data.head(3)

,draw_date,draw_num,winning_numbers,mega_ball,mega_ball_mod2,multiplier,pick5_1,pick5_1_zscore,pick5_2,pick5_2_zscore,pick5_3,pick5_3_zscore,pick5_4,pick5_4_zscore,pick5_5,pick5_5_zscore,pick5_sum,pick5_sum_zscore,pick5_mod2_sum,pick5_range
0,2024-05-17,40,08 17 40 60 70,3,1,2,8,-0.359521,17,-0.492509,40,0.431437,60,1.082709,70,1.123843,195,0.495525,1,62
1,2024-05-14,39,13 19 43 62 64,6,0,3,13,0.214504,19,-0.321895,43,0.664474,62,1.241801,64,0.537490,201,0.632157,3,51
2,2024-05-10,38,13 22 26 32 65,18,0,4,13,0.214504,22,-0.065975,26,-0.656070,32,-1.144578,65,0.635216,158,-0.347041,2,52


### Plot it

Let's employ a bar chart to visualize the frequency of the Pick 5 range values.

💡 This notebook features a [Vega-Altair](https://altair-viz.github.io/) bar chart. I've defined a function named `encode_mark_bar()` that standardizes the encoding of each plot. Implementing functions eliminates code duplication and enhances code modularization.

In [35]:
def encode_mark_bar(
    base,
    x_shorthand,
    x_axis_title,
    x_maxbins=15,
    x_scale_domain=(0, 305),
    y_shorthand="count():Q",
    y_axis_title="Count",
    y_scale_domain=(0, 30),
    color_shorthand="count():Q",
    color_title="Count",
    color_scale_scheme="blues",
):
    """Encode a bar chart with text annotations.

    Parameters:
        base (alt.Chart): The base chart to encode.
        x_shorthand (str): The x-axis shorthand (field, aggregate, type).
        x_axis_title (str): The x-axis title.
        x_maxbins (int): The maximum number of bins.
        x_scale_domain (tuple): The x-axis scale domain.
        y_shorthand (str): The y-axis shorthand (field, aggregate, type).
        y_axis_title (str): The y-axis title.
        y_scale_domain (tuple): The y-axis scale domain.
        color_shorthand (str): The color shorthand (field, aggregate, type).
        color_title (str): The color title.
        color_scale_scheme (str): The color scheme for the chart.

    Returns:
        alt.LayerChart: The encoded chart.
    """

    bar = base.mark_bar().encode(
        x=alt.X(
            shorthand=x_shorthand,
            axis=alt.Axis(
                labelAngle=-60,
                labelFontWeight="normal",
                tickCount=15,
                tickMinStep=5,
                title=x_axis_title,
                titleFontSize=10,
                titleFontWeight="normal",
            ),
            bin=alt.Bin(maxbins=x_maxbins),
            scale=alt.Scale(domain=x_scale_domain, padding=0.3),
        ),
        y=alt.Y(
            shorthand=y_shorthand,
            axis=alt.Axis(
                labelFontWeight="normal",
                title=y_axis_title,
                titleFontSize=10,
                titleFontWeight="normal",
            ),
            scale=alt.Scale(domain=y_scale_domain),
        ),
        color=alt.Color(
            shorthand=color_shorthand,
            legend=alt.Legend(
                gradientThickness=7,
                title=color_title,
                titleFontSize=10,
                titleFontWeight="bold",
                tickCount=5,
                tickMinStep=5,
                labelFontSize=8,
                labelFontWeight="normal",
            ),
            scale=alt.Scale(scheme=color_scale_scheme),
        ),
    )

    return bar

The code below instantiates an instance of an `alt.Chart()` object, calls the function `encode_mark_bar()` to encode the bar chart, and then adds additional encodings and labels to the chart. What does the chart suggest about Pick 5 winning number ranges and number selection strategies?

In [36]:
base = alt.Chart(data)
bar = encode_mark_bar(
    base,
    "pick5_range:Q",
    "Pick 5 range (frequency)",
    x_scale_domain=(0, 70),
    y_scale_domain=(0, 120),
    color_scale_scheme="oranges",
)
text = bar.mark_text(align="center", baseline="bottom", fontSize=8).encode(
    text="count():Q", color=alt.value("black")
)
rule = base.mark_rule().encode(
    alt.X("median(pick5_range):Q"),
    color=alt.value("red"),
    strokeWidth=alt.value(2),
)
chart = (bar + text + rule).properties(
    title=alt.Title("Pick 5 winning numbers range (frequency)", align="center"),
    width=300,
    height=300,
)
chart.show()

alt.LayerChart(...)

## Group-wise operations

You can also apply an arbitrary function group-wise and combine the results by calling the [`DataFrameGroupBy().apply()`](https://pandas.pydata.org/docs/reference/api/pandas.core.groupby.DataFrameGroupBy.apply.html) method. In line with the `DataFrame.apply()` and `Series.apply()` methods only the `func` argument is required; all other arguments are optional.

`DataFrameGroupBy.apply(func, *args, include_groups=True, **kwargs)`

In the example below, the Mega Millions winning draws are grouped by year and then a `lambda` function is applied to each group to compute the mean and the standard deviation of the "pick5_range" values.

In [37]:
mapper = data["draw_date"].dt.year
data.groupby(mapper, sort=False).size()
data.groupby(mapper, sort=False).apply(lambda x: x["pick5_range"].mean())
data.groupby(mapper, sort=False).apply(lambda x: x["pick5_range"].std())

draw_date
2024     40
2023    104
2022    104
2021    105
2020    104
2019    105
2018    104
2017     18
dtype: int64

draw_date
2024    48.675000
2023    47.846154
2022    47.884615
2021    46.961905
2020    45.461538
2019    47.428571
2018    48.076923
2017    47.666667
dtype: float64

draw_date
2024     9.653982
2023    12.687253
2022    12.481682
2021    12.230891
2020    12.652889
2019    10.547381
2018    12.419600
2017    13.894561
dtype: float64

All good, but when computing aggregations such as the mean, median, min, max, standard deviation, etc. look first to existing pandas methods before opting for the `apply()` method.

In [38]:
data.groupby(mapper, sort=False)["pick5_range"].mean()
data.groupby(mapper, sort=False)["pick5_range"].std()

draw_date
2024    48.675000
2023    47.846154
2022    47.884615
2021    46.961905
2020    45.461538
2019    47.428571
2018    48.076923
2017    47.666667
Name: pick5_range, dtype: float64

draw_date
2024     9.653982
2023    12.687253
2022    12.481682
2021    12.230891
2020    12.652889
2019    10.547381
2018    12.419600
2017    13.894561
Name: pick5_range, dtype: float64

### Mega Ball, Pick 5 overlap

How often does a winning draw feature a Mega Ball number (`1`-`25`) that also appears among the Pick 5 winning numbers (`1`-`70`)? We can return a count of the "overlap" by applying a `lambda` function along the column axis (i.e., row-wise) that evaluates whether or not a winning Mega Ball number can be found among its companion Pick 5 winning numbers.

In [39]:
mega_ball_pick5_overlap = data.apply(
    lambda x: x["mega_ball"]
    in x[["pick5_1", "pick5_2", "pick5_3", "pick5_4", "pick5_5"]].to_list(),
    axis=1,
)
mega_ball_pick5_overlap.value_counts(normalize=True)

False    0.919591
True     0.080409
Name: proportion, dtype: float64

The operation could be moved to a defined function if reuse is a consideration.

In [40]:
def is_mega_ball_in_pick5(series):
    """Check if the Mega Ball number matches a Pick 5 number row-wise.

    Parameters:
        series (pandas.Series): Represents a DataFrame row.

    Returns:
        bool: True if the Mega Ball winning number matches a Pick 5 winning number;
              otherwise False.
    """

    return (
        series["mega_ball"]
        in series[["pick5_1", "pick5_2", "pick5_3", "pick5_4", "pick5_5"]].to_list()
    )

In [41]:
mega_ball_pick5_overlap = data.apply(is_mega_ball_in_pick5, axis=1)
mega_ball_pick5_overlap.value_counts(normalize=True)

False    0.919591
True     0.080409
Name: proportion, dtype: float64

### Annual counts

Let's assume that we need to track the annual incidence of Mega Ball/Pick 5 overlap. This will require us to group the winning draws by year and then apply either the `is_mega_ball_in_pick5()` function or the matching `lambda` function to __each row__ within each group.

❗ We need to be careful here. Calling `DataFrameGroupBy.apply()` and passing it either the defined or `lambda` function along with the keyword argument `axis=1` will trigger a `TypeError: is_mega_ball_in_pick5() got an unexpected keyword argument 'axis'`.

Why? Because the method applies the function __group-wise__ not __row-wise__ and the keyword argument `axis=1` is passed on to the function as an optional (and erroneous) argument.

Instead, we need to nest a `DataFrame.apply()` method call inside the outer `DataFrameGroupBy.apply()` call to ensure that the necessary row-wise operation is performed.

In [42]:
mapper = data["draw_date"].dt.year  # repeated for clarity

# WARN: The following expression triggers a runtime exception
# data.groupby(mapper, sort=False).apply(is_mega_ball_in_pick5, axis=1).value_counts()

data.groupby(mapper, sort=False).apply(
    lambda x: x.apply(is_mega_ball_in_pick5, axis=1).value_counts()
)

count,False,True
draw_date,,
2024,38,2
2023,99,5
2022,94,10
2021,95,10
2020,98,6
2019,97,8
2018,91,13
2017,17,1


In [43]:
data.groupby(mapper, sort=False).apply(
    lambda x: x.apply(
        lambda x: x["mega_ball"]
        in x[["pick5_1", "pick5_2", "pick5_3", "pick5_4", "pick5_5"]].to_list(),
        axis=1,
    ).value_counts()  # inside apply
)

count,False,True
draw_date,,
2024,38,2
2023,99,5
2022,94,10
2021,95,10
2020,98,6
2019,97,8
2018,91,13
2017,17,1


### Write to file

Let's conclude our discussion of the `Series.apply()`, `DataFrame.apply()`, and `DataFrameGroupBy.apply()` methods by writing `data` to a CSV file. We will make use of the data file in a future lesson. 🙂

In [44]:
filepath = "./data/mega_millions-apply-2017_24.csv"
data.to_csv(filepath, index=False)